<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session5/1-Text-generation-GPT.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 5 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Generación de texto - GPT

En este notebook se aborda el problema de generación de texto aplicado a la creación de contenido en español a partir de un corpus de noticias. Para ello, se utiliza un conjunto de datos compuesto por textos periodísticos, los cuales permiten entrenar un modelo capaz de aprender patrones lingüísticos, estructura narrativa y estilo propio de este tipo de contenido.

El objetivo principal es desarrollar un modelo capaz de generar texto coherente y contextualmente relevante a partir de una entrada inicial (prompt). Para lograrlo, se emplea el modelo GPT-2, una arquitectura basada en transformers de tipo autoregresivo, que predice la siguiente palabra en una secuencia dada. A diferencia de modelos bidireccionales como BERT, GPT-2 procesa el lenguaje de manera unidireccional, lo que lo hace especialmente adecuado para tareas de generación de texto, permitiendo producir contenido fluido y consistente.

## 0. Configuración del Entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las reseñas.

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

## 1. Modelo GPT-2 en Español — Generación Base (sin fine-tuning)

### ¿Qué es GPT?

Los modelos GPT (*Generative Pre-trained Transformer*) usan bloques de **Transformer Decoder** encadenados. Su tarea principal es predecir el **siguiente token** dado un contexto anterior (modelado de lenguaje causal). Esto contrasta con BERT, que usa encoders y predice tokens enmascarados en posiciones arbitrarias.

La arquitectura se basa en el mecanismo de **self-attention causal**: cada token solo puede "ver" los tokens anteriores, lo cual es fundamental para la generación secuencial de texto.

Usaremos `DeepESP/gpt2-spanish`, un modelo GPT-2 pre-entrenado en español con mayor cobertura del idioma que `mrm8488/spanish-gpt2`.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "DeepESP/gpt2-spanish"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

model.resize_token_embeddings(len(tokenizer))

model

### 1.1 Exploración del Tokenizador

Antes de generar texto, es importante entender cómo el tokenizador descompone el texto en *tokens*. GPT-2 usa **Byte-Pair Encoding (BPE)**: un algoritmo que divide palabras en sub-unidades frecuentes aprendidas del corpus de entrenamiento.



In [ ]:
import pandas as pd

# Observemos cómo el tokenizador descompone texto en español
ejemplos = [
    "Muchos años después, frente al pelotón de fusilamiento",
    "En el principio era el Verbo",
    "El coronel no tiene quien le escriba"
]

for texto in ejemplos:
    tokens = tokenizer.tokenize(texto)
    ids = tokenizer.encode(texto)
    print(f'Texto : "{texto}"')
    print(f'Tokens: {tokens}')
    print(f'IDs   : {ids}')
    print(f'Longitud: {len(tokens)} tokens\n')

### 1.2 Generación de Texto: Exploración de Estrategias de Decodificación

La forma en que se selecciona el siguiente token define el comportamiento de la generación. Existen dos grandes enfoques:

1. **Greedy decoding**: siempre elige el token de mayor probabilidad → resultados deterministas pero repetitivos.
2. **Sampling**: muestrea de la distribución de probabilidad → más creativo pero potencialmente incoherente.
   - **Temperature**: escala las probabilidades. Valores bajos (<1) → más conservador; valores altos (>1) → más aleatorio.
   - **Top-K**: muestrea solo entre los K tokens más probables.
   - **Top-P (Nucleus)**: muestrea del conjunto mínimo de tokens que acumulan probabilidad ≥ P.


In [ ]:
def generate_text(model, tokenizer, prompt, max_length=100, device='cpu', **gen_kwargs):
    """
    Función auxiliar para generar texto a partir de un prompt.
    Recibe parámetros adicionales de generación (**gen_kwargs) para
    facilitar la comparación entre estrategias de decodificación.
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Prompt inicial en estilo literario latinoamericano
prompt = "Muchos años después, frente al pelotón de fusilamiento"
print(f'Prompt: "{prompt}"\n')

In [ ]:
# ---- Estrategia 1: Greedy Decoding ----
# Siempre elige el token de mayor probabilidad. Determinista y repetitivo.
texto_greedy = generate_text(model, tokenizer, prompt, max_length=100, device=device, do_sample=False)
print('--- GREEDY DECODING ---')
print(texto_greedy)
print()

In [ ]:
# ---- Estrategia 2: Sampling con Temperature baja (0.5) ----
# Temperature baja = distribución más 'afilada', menor aleatoriedad
texto_temp_baja = generate_text(model, tokenizer, prompt, max_length=100, device=device,
                                 do_sample=True, temperature=0.5)
print('--- SAMPLING: Temperature=0.5 (conservador) ---')
print(texto_temp_baja)
print()

In [ ]:
# ---- Estrategia 3: Sampling con Temperature alta (1.2) ----
# Temperature alta = distribución más 'plana', mayor creatividad pero más riesgo de incoherencia
texto_temp_alta = generate_text(model, tokenizer, prompt, max_length=100, device=device,
                                 do_sample=True, temperature=1.2)
print('--- SAMPLING: Temperature=1.2 (creativo) ---')
print(texto_temp_alta)
print()

In [ ]:
# ---- Estrategia 4: Top-K Sampling (K=50) ----
# Solo muestrea entre los 50 tokens más probables en cada paso
texto_topk = generate_text(model, tokenizer, prompt, max_length=100, device=device,
                            do_sample=True, top_k=50, temperature=0.8)
print('--- TOP-K SAMPLING: K=50, temperature=0.8 ---')
print(texto_topk)
print()

In [ ]:
# ---- Estrategia 5: Nucleus Sampling (Top-P = 0.92) ----
# Muestrea del conjunto mínimo de tokens que acumulan P=92% de probabilidad
# Esta es generalmente la estrategia recomendada para texto más natural
texto_nucleus = generate_text(model, tokenizer, prompt, max_length=100, device=device,
                               do_sample=True, top_p=0.92, top_k=0, temperature=0.8)
print('--- NUCLEUS SAMPLING: P=0.92, temperature=0.8 ---')
print(texto_nucleus)
print()

In [ ]:
# ---- Tabla comparativa de estrategias de decodificación ----
# Generamos múltiples textos con diferentes estrategias y los comparamos en un DataFrame

import textwrap

resultados = [
    {"Estrategia": "Greedy",                  "Parámetros": "do_sample=False",              "Texto generado": texto_greedy[len(prompt):][:120].strip()},
    {"Estrategia": "Temperature baja",        "Parámetros": "temp=0.5",                     "Texto generado": texto_temp_baja[len(prompt):][:120].strip()},
    {"Estrategia": "Temperature alta",        "Parámetros": "temp=1.2",                     "Texto generado": texto_temp_alta[len(prompt):][:120].strip()},
    {"Estrategia": "Top-K",                   "Parámetros": "K=50, temp=0.8",               "Texto generado": texto_topk[len(prompt):][:120].strip()},
    {"Estrategia": "Nucleus (Top-P)",         "Parámetros": "P=0.92, temp=0.8",             "Texto generado": texto_nucleus[len(prompt):][:120].strip()},
]

df_resultados = pd.DataFrame(resultados)
pd.set_option('display.max_colwidth', 200)
print('COMPARACIÓN DE ESTRATEGIAS DE DECODIFICACIÓN (modelo base, sin fine-tuning)')
df_resultados

**Observaciones sobre el modelo base:**
- El **greedy decoding** tiende a producir texto repetitivo y estereotipado, ya que siempre toma el camino de mayor probabilidad sin diversidad.
- La **temperatura baja** produce texto más fluido y gramaticalmente correcto, pero a veces genérico.
- La **temperatura alta** introduce más variedad léxica pero puede perder coherencia narrativa.
- **Top-K** limita el vocabulario activo, lo que puede producir texto más cohesionado pero menos sorpresivo.
- **Nucleus sampling** tiende a ofrecer el mejor balance entre coherencia y creatividad, adaptando dinámicamente el conjunto de tokens candidatos según el contexto.

El texto generado con el modelo base tiene un estilo genérico, sin el característico tono literario latinoamericano. Esto motivará el fine-tuning en la siguiente sección.

## 2. Análisis Exploratorio del Dataset para Fine Tuning — Corpus News

Para el fine-tuning usaremos el dataset `MarcOrfilaCarreras/spanish-news` de Hugging Face, que contiene textos periodísticos en español.

 Un corpus de noticias presenta frases más largas, vocabulario más rico y estructuras sintácticas más complejas, lo que permite explorar el impacto del fine-tuning en dimensiones lingüísticas más profundas.

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import re


dataset = load_dataset("MarcOrfilaCarreras/spanish-news")
dataset

In [ ]:
dataset.set_format('pandas')

df = dataset['train'].to_pandas()
print(f'Filas cargadas: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

In [ ]:
df['Palabras por noticia'] = df['text'].str.split().apply(len)
df['Palabras por noticia'].median()

In [ ]:
# Seleccionamos y limpiamos la columna de texto
# El dataset tiene una columna 'text' o 'content' con el texto principal

# Identificamos la columna de texto correcta
col_texto = 'text' if 'text' in df.columns else df.columns[0]
print(f'Columna de texto utilizada: {col_texto}')

# Limpieza básica del texto
def limpiar_texto(texto):
    """Limpia caracteres extraños y espacios múltiples"""
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'\s+', ' ', texto)  # Elimina espacios múltiples
    texto = texto.strip()
    return texto

df['texto_limpio'] = df[col_texto].apply(limpiar_texto)

# Filtramos textos muy cortos (< 50 palabras) o muy largos (> 300 palabras)
df['n_palabras'] = df['texto_limpio'].str.split().apply(len)
df_filtrado = df[(df['n_palabras'] >= 50) & (df['n_palabras'] <= 600)].copy()
df_filtrado = df_filtrado.reset_index(drop=True)

print(f'Textos después del filtrado: {len(df_filtrado)}')
print(f'Rango de palabras: {df_filtrado["n_palabras"].min()} - {df_filtrado["n_palabras"].max()}')
print(f'Mediana de palabras por texto: {df_filtrado["n_palabras"].median()}')
df_filtrado[['texto_limpio', 'n_palabras']].head(5)

In [ ]:
# ---- Visualizaciones del corpus ----

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis Exploratorio del Corpus', fontsize=14, fontweight='bold')

# ---- Plot 1: Distribución de longitud de textos ----
axes[0].hist(df_filtrado['n_palabras'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df_filtrado['n_palabras'].median(), color='tomato', linestyle='--', linewidth=2,
                label=f"Mediana: {df_filtrado['n_palabras'].median():.0f} palabras")
axes[0].axvline(df_filtrado['n_palabras'].mean(), color='orange', linestyle='--', linewidth=2,
                label=f"Media: {df_filtrado['n_palabras'].mean():.0f} palabras")
axes[0].set_xlabel('Número de palabras')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de longitud de los textos')
axes[0].legend()

# ---- Plot 2: Distribución de longitud de oraciones ----
# Calculamos el número de oraciones por texto
df_filtrado['n_oraciones'] = df_filtrado['texto_limpio'].apply(
    lambda x: len(re.split(r'[.!?]+', x))
)
axes[1].hist(df_filtrado['n_oraciones'], bins=30, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[1].axvline(df_filtrado['n_oraciones'].median(), color='tomato', linestyle='--', linewidth=2,
                label=f"Mediana: {df_filtrado['n_oraciones'].median():.0f} oraciones")
axes[1].set_xlabel('Número de oraciones')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de oraciones por texto')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ----  Nube de palabras del corpus ----
try:
    from wordcloud import WordCloud, STOPWORDS

    # Stopwords en español
    stopwords_es = set(STOPWORDS) | {
        'de', 'la', 'el', 'en', 'y', 'a', 'los', 'se', 'del', 'las', 'un', 'por',
        'con', 'una', 'su', 'al', 'lo', 'como', 'más', 'que', 'pero', 'sus', 'le',
        'ya', 'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin',
        'sobre', 'también', 'me', 'hasta', 'hay', 'donde', 'quien', 'desde', 'todo',
        'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese', 'eso',
        'ante', 'ellos', 'e', 'esto', 'mí', 'antes', 'algunos', 'qué', 'unos', 'yo',
        'otro', 'otras', 'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada',
    }

    corpus_unido = ' '.join(df_filtrado['texto_limpio'].sample(min(500, len(df_filtrado))).tolist())

    wc = WordCloud(
        width=800, height=400,
        background_color='white',
        stopwords=stopwords_es,
        max_words=100,
        colormap='viridis',
        collocations=False
    ).generate(corpus_unido)

    plt.figure(figsize=(12, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title('Palabras más frecuentes en el corpus (sin stopwords)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('wordcloud_corpus.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Nube de palabras guardada como wordcloud_corpus.png')

except ImportError:
    print('WordCloud no está instalado. Instálalo con: pip install wordcloud')

**Hallazgo del EDA:**
- pendiente


## 3. Preparación del Dataset para Fine-Tuning

In [ ]:
from datasets import load_dataset

dataset = load_dataset("MarcOrfilaCarreras/spanish-news")
dataset

In [ ]:
dataset['train'][0]

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['text'], max_length=max_len, truncation=True)
    return _preprocess_function

In [ ]:
dataset.reset_format()
tokenized_dataset = dataset['train'].map(preprocess_function(max_len=64), batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in ['input_ids', 'attention_mask']])
tokenized_dataset = tokenized_dataset.train_test_split(train_size=0.9)
tokenized_dataset.set_format('torch')
tokenized_dataset

In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments


batch_size = 32 if IN_COLAB else 2
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf-gpt',
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='none'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
)

In [ ]:
%%time
trainer.train()

# 8. Conclusiones

En este notebook se evaluaron cuatro enfoques para la clasificación de textos relacionados con cambio climático usando el dataset somosnlp/spa_climate_detection: BERT congelado, BERTIN RoBERTa congelado, BERT con clasificador personalizado y BERT con fine-tuning completo.

### Principales hallazgos

- Los modelos congelados funcionan como una línea base razonable, pero muestran limitaciones claras en capacidad de discriminación.
- BERTIN RoBERTa supera levemente a BERT en modo congelado, lo que sugiere una ventaja del preentrenamiento y del tokenizador para textos en español.
- Añadir un clasificador más expresivo mejora el rendimiento respecto a los modelos congelados, pero no elimina por completo el sesgo entre clases.
- El fine-tuning completo obtiene el mejor desempeño global y el mejor equilibrio entre la clase "clima" y la clase "no clima".

### Interpretación de resultados

El análisis de métricas por clase muestra que los modelos congelados tienden a favorecer la clase positiva, logrando recalls muy altos para "clima" pero bastante más bajos para "no clima". Esto indica que, aunque los embeddings preentrenados capturan bien el vocabulario temático, no son suficientes por sí solos para resolver adecuadamente los casos más ambiguos.

El clasificador personalizado reduce parte de esta limitación al introducir una cabeza no lineal más potente. Sin embargo, el principal salto de calidad aparece cuando se permite que todo el modelo ajuste sus representaciones al dominio específico. En ese escenario, BERT no solo mejora la accuracy, sino también el balance entre clases, lo que fortalece la capacidad de generalización del sistema.

### Conclusión final

Los resultados demuestran que, para esta tarea, el factor más importante no es únicamente elegir un buen modelo preentrenado, sino permitir que dicho modelo se adapte a la tarea mediante fine-tuning. Por ello, el enfoque más sólido en este notebook es BERT con fine-tuning completo, mientras que los modelos congelados y el clasificador personalizado sirven como comparaciones útiles para evidenciar cómo cambia el rendimiento al aumentar el nivel de adaptación del modelo.